# 03 · Doctor data: Symcat, case cards, consultations, and doctor-model training strings

**SimClinic, CS425 Group 7 (doctor model).** This notebook prepares everything the doctor model and its baselines need:

1. `data/processed/doctor/symcat_conditions.json`: Symcat parsed from the raw export, with symptom percentages, synonyms and age/sex odds.
2. `data/processed/doctor/cards/{split}.jsonl`: case cards (hidden disease, 3–6 symptoms, persona), split 80/10/10 by card and stratified by disease.
3. `data/processed/doctor/dialogues/{split}.jsonl`: multi-turn consultations ending in a top-3 differential.
4. `data/processed/doctor/tokenized/<model_slug>/`: the tokenized `DatasetDict`, with loss **only on the doctor's turns and diagnosis**.

**Data.** The case cards and the LLM-generated consultations come from notebook 07 (`data/processed/cards/cards.jsonl`, `data/processed/dialogues/dialogues.jsonl`): 61 conditions, about 2,000 cards and 5,000 dialogues. With `CARDS_JSONL` and `DIALOGUES_JSONL` set to `None`, this notebook instead builds the earlier **stand-in** data:
- **Cards:** sampled from Symcat percentages for the same 61 conditions.
- **Dialogues:** a naive Bayes doctor that asks the most informative symptom question, talking to a rule-based patient that answers from the card.

The stand-in dialogues are templated. They are useful for testing the pipeline end to end without a GPU, but they are not the training data.

Training string format (doctor role, so the hidden header is left out):
```
<|doctor|>What brings you in today?<|patient|>I've had a fever ...<|doctor|>Any cough?<|patient|>No.<|doctor|>Thanks.<|diagnosis|>Flu; Common cold; Strep throat<|endoftext|>
└marker┘└──── loss ────────────┘└ loss ┘└──── masked ────────┘└marker┘└ loss ─┘└loss┘└ masked ┘└marker┘└─────────────── loss ─────────────────────────────────────┘
```
The doctor ends a question by emitting `<|patient|>`. It closes the consultation by emitting `<|diagnosis|>`.

## 0. Config

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from simclinic import cards as C

# --- model -------------------------------------------------------------------
MODEL_NAME = "gpt2"          # gpt2 and gpt2-medium share a tokenizer, so one tokenized set serves both
MAX_LENGTH = 1024            # full GPT-2 context; longer dialogues are dropped (never truncated: the diagnosis is at the end)

# --- data sources (notebook 07; None = build the stand-in) -------------------------
CARDS_JSONL = ROOT / "data/processed/cards/cards.jsonl"
DIALOGUES_JSONL = ROOT / "data/processed/dialogues/dialogues.jsonl"
LEXICON_JSON = None          # None = simclinic/symptom_lexicon.json

# --- stand-in generation -----------------------------------------------------------
CONDITIONS = C.CONDITIONS    # 59 Symcat conditions + the team's dengue and HFMD cards
CARDS_PER_CONDITION = 33     # 61 x 33 = 2,013 cards, as in the proposal (~2,000)
DIALOGUES_PER_CARD = {"train": 3, "dev": 1, "test": 1}   # ~5,000 training dialogues
MAX_QUESTIONS = 10

# --- paths ---------------------------------------------------------------------
RAW_SYMCAT = ROOT / "data" / "raw" / "symcat"
OUT_DIR = ROOT / "data" / "processed" / "doctor"
SPLITS = ["train", "dev", "test"]
SEED = 42

## 1. Parse Symcat and re-check the audit
The raw CSVs come from the symcat-to-synthea repository and are downloaded if they are missing. Our own parser (`simclinic/symcat.py`) must reproduce the numbers in the proposal's audit table.

In [2]:
import json
from collections import Counter
from dataclasses import asdict
import pandas as pd
from simclinic import symcat

pd.set_option("display.max_colwidth", 120)

symcat.download(RAW_SYMCAT)
conditions = symcat.parse_conditions(RAW_SYMCAT / "symcat-801-diseases.csv")
n_links = sum(len(c.symptoms) for c in conditions.values())
n_symptoms = symcat.count_symptoms(RAW_SYMCAT / "symcat-474-symptoms.csv")
capped = sum(len(c.symptoms) == 12 for c in conditions.values())
tied = sum(bool(c.symptoms) and max(Counter(c.symptoms.values()).values()) >= 5 for c in conditions.values())
passing = [n for n, c in conditions.items() if symcat.passes_quality_filter(c)]

audit = pd.Series({"conditions": len(conditions), "symptoms": n_symptoms, "condition-symptom links": n_links,
                   "conditions with exactly 12 symptoms": capped, "conditions with >= 5 tied percentages": tied,
                   "conditions passing the quality filter": len(passing)})
assert audit.tolist() == [801, 474, 9162, 687, 296, 391], audit
audit.to_frame("value")

,value
conditions,801
symptoms,474
condition-symptom links,9162
conditions with exactly 12 symptoms,687
conditions with >= 5 tied percentages,296
conditions passing the quality filter,391


In [3]:
# Why the filter matters: a well-described condition next to one that comes from a handful of records.
pd.DataFrame({n: [f"{s} {round(p * 100)}%" for s, p in conditions[n].symptoms.items()] for n in ["Flu", "Dengue fever"]})

,Flu,Dengue fever
0,Fever 91%,Sore throat 100%
1,Cough 87%,Fever 100%
2,Sore throat 63%,Shoulder cramps or spasms 32%
3,Nasal congestion 63%,Ankle pain 32%
4,Ache all over 56%,Wrist pain 32%
5,Flu-like syndrome 52%,Pain during pregnancy 32%
6,Headache 52%,Joint stiffness or tightness 32%
7,Vomiting 47%,Excessive anger 32%
8,Chills 38%,Pain or soreness of breast 32%
9,Coryza 37%,Knee lump or mass 32%


## 2. Condition set and symptom lexicon
The lexicon (notebook 07) maps each Symcat symptom to lay phrasings ("Coryza" → "a runny nose"). It is used in three places:
- the stand-in patient's answers;
- reading which symptoms a dialogue revealed, for the naive Bayes baseline;
- the dialogue quality filter.

Every symptom of the chosen conditions must have an entry.

In [4]:
from simclinic.lexicon import Lexicon

conditions = C.load_conditions(RAW_SYMCAT / "symcat-801-diseases.csv")   # adds the team's dengue and HFMD cards
passing = {C.RENAMED.get(n, n) for n in passing}                       # Symcat misspells BPPV
missing = [n for n in CONDITIONS if n not in passing and n not in C.TEAM_CONDITIONS]
assert not missing, f"conditions failing the Symcat quality filter: {missing}"
symptom_vocab = sorted({s for n in CONDITIONS for s in conditions[n].symptoms})
lexicon_raw = json.loads(Path(LEXICON_JSON or Path(C.__file__).with_name("symptom_lexicon.json")).read_text())
no_entry = [s for s in symptom_vocab if s not in lexicon_raw]
lexicon = Lexicon.load(LEXICON_JSON, symptoms=symptom_vocab)

print(f"{len(CONDITIONS)} conditions, {len(symptom_vocab)} distinct symptoms, "
      f"{sum(len(lexicon.phrases[s]) for s in symptom_vocab)} lay phrasings; without an entry: {no_entry or 'none'}")
pd.DataFrame([(s, lexicon.lay(s), ", ".join(lexicon.phrases[s][1:4])) for s in
              ["Coryza", "Nasal congestion", "Ache all over", "Painful urination", "Suprapubic pain", "Dizziness"]],
             columns=["Symcat symptom", "used in text", "also matched (first 3)"])

61 conditions, 202 distinct symptoms, 1509 lay phrasings; without an entry: none


,Symcat symptom,used in text,also matched (first 3)
0,Coryza,a runny nose,"runny nose, nose is running, nose running"
1,Nasal congestion,a blocked nose,"blocked nose, stuffy nose, stuffed nose"
2,Ache all over,body aches,"body ache, aching all over, aches all over"
3,Painful urination,pain when peeing,"burning when peeing, stinging when peeing, painful peeing"
4,Suprapubic pain,pain just above the pubic bone,"pain above the pubic bone, pain in the bladder area, bladder pain"
5,Dizziness,dizziness,"dizzy, light-headed, lightheaded"


## 3. Case cards
Stand-in cards draw each symptom independently with its Symcat probability and resample until there are 3–6 symptoms. Age and sex are weighted by Symcat's odds ratios, using adult age groups only. Ethnicity follows Singapore's resident population shares. The split is 80/10/10 per disease.

In [5]:
import random

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

if CARDS_JSONL:
    cards = read_jsonl(CARDS_JSONL)
    if "split" not in cards[0]:
        cards = C.split_cards(cards, seed=SEED)
else:
    cards = C.split_cards(C.make_cards(conditions, CONDITIONS, CARDS_PER_CONDITION, seed=SEED), seed=SEED)
card_by_id = {c["card_id"]: c for c in cards}

assert all(set(c["symptoms"]) <= set(conditions[c["disease"]].symptoms) for c in cards), "card symptom not in Symcat list"
per_split = pd.DataFrame(cards).groupby("split").agg(cards=("card_id", "size"), diseases=("disease", "nunique"))
assert (per_split["diseases"] == len(CONDITIONS)).all(), "every disease must appear in every split"
per_split

,cards,diseases
split,,
dev,201,61
test,201,61
train,1611,61


In [6]:
cards[0]

{'card_id': 'flu-000',
 'disease': 'Flu',
 'symptoms': ['Fever',
  'Cough',
  'Sore throat',
  'Nasal congestion',
  'Headache',
  'Coryza'],
 'persona': {'age': 20,
  'sex': 'M',
  'ethnicity': 'Malay',
  'occupation': 'software engineer',
  'temperament': 'anxious',
  'health_literacy': 'medium',
  'onset': 'this morning',
  'severity': 'moderate'},
 'split': 'train'}

In [7]:
persona = pd.DataFrame([c["persona"] for c in cards])
pd.concat({k: persona[k].value_counts(normalize=True).round(2) for k in ["sex", "ethnicity", "temperament"]}).to_frame("share").T

sex      ethnicity                     temperament                     
         F    M   Chinese Malay Indian Others        calm anxious terse chatty
share  0.5  0.5      0.73  0.15   0.08   0.04        0.26    0.25  0.25   0.24

## 4. Consultations
Stand-in dialogues come from `NBDoctor` talking to `CardPatient` (both in `simclinic/`):
- **NBDoctor.** After an opener and sometimes an onset question, it asks about the symptom with the highest expected information gain under Symcat naive Bayes. Sometimes it asks two at once. It diagnoses once one condition reaches 70% posterior, after at least 3 questions, or after 10.
- **CardPatient.** It answers yes or no from its card in lay words, with a little temperament flavour.
- **Training label.** The true disease comes first, followed by the doctor's two most likely alternatives. The model therefore learns *P(disease | dialogue)* instead of copying naive Bayes.

Every dialogue then goes through the proposal's filter: the patient must not name its disease or claim a symptom that is not on the card.

In [8]:
from simclinic.consult import run_consultation
from simclinic.doctors import NBDoctor
from simclinic.naive_bayes import SymcatNaiveBayes
from simclinic.patients import CardPatient

nb = SymcatNaiveBayes({n: conditions[n].symptoms for n in CONDITIONS})

def standin_dialogues(card, n, seed):
    out = []
    for k in range(n):
        s = seed * 10 + k
        rec = run_consultation(NBDoctor(nb, lexicon, max_questions=MAX_QUESTIONS, seed=s),
                               CardPatient(card, lexicon, seed=s), max_questions=MAX_QUESTIONS)
        others = [d for d in rec["diagnosis"] if d != card["disease"]][:2]
        out.append({"dialogue_id": f"{card['card_id']}-d{k}", "card_id": card["card_id"], "split": card["split"],
                    "disease": card["disease"], "turns": rec["turns"], "diagnosis": [card["disease"], *others],
                    "nb_doctor_diagnosis": rec["diagnosis"], "source": "standin-nb-doctor"})
    return out

if DIALOGUES_JSONL:
    dialogues = read_jsonl(DIALOGUES_JSONL)
else:
    dialogues = [d for i, c in enumerate(cards) for d in standin_dialogues(c, DIALOGUES_PER_CARD[c["split"]], i)]

names = {n: C.disease_names(conditions[n]) for n in CONDITIONS}
rejected = []
for d in dialogues:
    problems = C.dialogue_problems(d, card_by_id[d["card_id"]], lexicon, names[d["disease"]])
    if problems:
        rejected.append({"dialogue_id": d["dialogue_id"], "problems": problems})
bad = {r["dialogue_id"] for r in rejected}
dialogues = [d for d in dialogues if d["dialogue_id"] not in bad]
print(f"{len(dialogues):,} dialogues kept, {len(rejected)} rejected by the filter")
pd.Series(Counter(d["split"] for d in dialogues)).to_frame("dialogues").T

5,229 dialogues kept, 0 rejected by the filter


,train,dev,test
dialogues,4827,201,201


In [9]:
def show(d):
    card = card_by_id[d["card_id"]]
    print(f"[{d['dialogue_id']}] {card['disease']} | card: {', '.join(card['symptoms'])} | {card['persona']['temperament']}")
    for t in d["turns"]:
        print(f"  {t['role']:>7}: {t['text']}")
    print(f"  diagnosis label: {'; '.join(d['diagnosis'])}\n")

for d in random.Random(SEED).sample(dialogues, 3):
    show(d)

[chickenpox-020-d1] Chickenpox | card: Skin rash, Fever, Cough | anxious
   doctor: What brings you in today?
  patient: I've got a rash and a bit of a fever.
   doctor: When did the rash start?
  patient: It started last night, just after I went to bed.
   doctor: Where on your body is the rash?
  patient: It's all over my body, but mostly on my arms and legs.
   doctor: Does it feel itchy or painful?
  patient: It doesn't feel itchy, just uncomfortable. I didn't feel it was painful until now.
   doctor: Have you had any coughing or other symptoms?
  patient: Yeah, I've got a cough too, just a little one.
   doctor: Do you have any other symptoms like a blocked nose or sore throat?
  patient: No, I don't have a blocked nose or sore throat. I'm just worried about what it could be.
   doctor: Thank you for letting me know about your symptoms. We'll need to keep an eye on how you're doing.
  diagnosis label: Chickenpox; Viral exanthem; Herpangina

[acute-sinusitis-020-d0] Acute sinusitis

In [10]:
n_q = pd.Series([sum(t["role"] == "doctor" for t in d["turns"]) - 1 for d in dialogues])
stats = {
    "questions per dialogue (mean)": n_q.mean().round(2),
    "questions per dialogue (min / max)": f"{n_q.min()} / {n_q.max()}",
    "words per patient turn (median)": pd.Series([len(t["text"].split()) for d in dialogues for t in d["turns"] if t["role"] == "patient"]).median(),
}
if "nb_doctor_diagnosis" in dialogues[0]:
    stats["NB doctor top-1 / top-3 (for reference)"] = (
        f"{pd.Series([d['nb_doctor_diagnosis'][0] == d['disease'] for d in dialogues]).mean():.3f} / "
        f"{pd.Series([d['disease'] in d['nb_doctor_diagnosis'] for d in dialogues]).mean():.3f}")
pd.Series(stats).to_frame("value")

,value
questions per dialogue (mean),6.66
questions per dialogue (min / max),4 / 12
words per patient turn (median),10.0


## 5. Save cards, dialogues and the parsed Symcat table

In [11]:
def write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

for s in SPLITS:
    write_jsonl(OUT_DIR / "cards" / f"{s}.jsonl", [c for c in cards if c["split"] == s])
    write_jsonl(OUT_DIR / "dialogues" / f"{s}.jsonl", [d for d in dialogues if d["split"] == s])
write_jsonl(OUT_DIR / "dialogues" / "rejected.jsonl", rejected)
(OUT_DIR / "symcat_conditions.json").write_text(
    json.dumps({n: asdict(c) for n, c in conditions.items()}, indent=1, ensure_ascii=False), encoding="utf-8")
(OUT_DIR / "conditions.json").write_text(json.dumps(CONDITIONS, indent=1), encoding="utf-8")
print("saved ->", OUT_DIR)

saved -> /home/kth/Documents/Github/project/model/data/processed/doctor


## 6. Doctor training strings
`format_doctor_example` (in `simclinic/format.py`, which inference uses too) returns the string and the character spans to train on. Labels are `-100` on every token that does not start inside a span. The special tokens and their order match notebook 01, so the patient and doctor models share one vocabulary.

In [12]:
from transformers import AutoTokenizer
from simclinic.format import ROLE_TOKENS, EXTRA_SPECIAL_TOKENS, DIAGNOSIS_TOKEN, format_doctor_example

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
assert tokenizer.is_fast, "a fast tokenizer is needed for offset mapping"
base_vocab = len(tokenizer)
tokenizer.add_special_tokens({"additional_special_tokens": list(ROLE_TOKENS.values()) + EXTRA_SPECIAL_TOKENS})
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
MAX_LEN = min(MAX_LENGTH, tokenizer.model_max_length)
MODEL_SLUG = MODEL_NAME.replace("/", "__")
print(f"{MODEL_NAME}: vocab {base_vocab} -> {len(tokenizer)}; max_len={MAX_LEN}")

demo_text, demo_spans = format_doctor_example(dialogues[0]["turns"], dialogues[0]["diagnosis"], tokenizer.eos_token)
print(demo_text, "\n")
print("trained spans:", [demo_text[a:b] for a, b in demo_spans])

gpt2: vocab 50257 -> 50263; max_len=1024
<|doctor|>What brings you in today?<|patient|>I've got a cough<|doctor|>When did the cough start?<|patient|>Yesterday<|doctor|>Any fever?<|patient|>Yeah, a bit<|doctor|>Sore throat?<|patient|>Yeah, that too<|doctor|>Shortness of breath?<|patient|>Sometimes, yeah<|doctor|>Running nose?<|patient|>Yeah, it's runny<|doctor|>Coughing up phlegm?<|patient|>I've got coughing up phlegm too<|doctor|>Thanks for your time today.<|diagnosis|>Acute bronchitis; Pneumonia; Acute sinusitis<|endoftext|> 

trained spans: ['What brings you in today?<|patient|>', 'When did the cough start?<|patient|>', 'Any fever?<|patient|>', 'Sore throat?<|patient|>', 'Shortness of breath?<|patient|>', 'Running nose?<|patient|>', 'Coughing up phlegm?<|patient|>', 'Thanks for your time today.<|diagnosis|>Acute bronchitis; Pneumonia; Acute sinusitis<|endoftext|>']


In [13]:
def encode(d):
    text, spans = format_doctor_example(d["turns"], d["diagnosis"], tokenizer.eos_token)
    enc = tokenizer(text, return_offsets_mapping=True)
    labels = [tok if (e > s and any(a <= s < b for a, b in spans)) else -100
              for tok, (s, e) in zip(enc["input_ids"], enc["offset_mapping"])]
    return {"dialogue_id": d["dialogue_id"], "input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
            "labels": labels, "n_train_tokens": sum(l != -100 for l in labels)}

encoded = {s: [encode(d) for d in dialogues if d["split"] == s] for s in SPLITS}
too_long = {s: [e["dialogue_id"] for e in encoded[s] if len(e["input_ids"]) > MAX_LEN] for s in SPLITS}
encoded = {s: [e for e in encoded[s] if len(e["input_ids"]) <= MAX_LEN] for s in SPLITS}

pd.DataFrame({s: {
    "examples": len(encoded[s]),
    "dropped (> max_len)": len(too_long[s]),
    "tokens p50": int(pd.Series([len(e["input_ids"]) for e in encoded[s]]).median()),
    "tokens p95": int(pd.Series([len(e["input_ids"]) for e in encoded[s]]).quantile(.95)),
    "tokens max": max(len(e["input_ids"]) for e in encoded[s]),
    "trainable tokens": sum(e["n_train_tokens"] for e in encoded[s]),
} for s in SPLITS}).T

,examples,dropped (> max_len),tokens p50,tokens p95,tokens max,trainable tokens
train,4827,0,203,286,458,491089
dev,201,0,199,291,390,20302
test,201,0,194,278,356,19961


In [14]:
def show_mask(e, limit=200):
    """Trained tokens are shown in «», masked tokens plain."""
    print("".join(f"«{tokenizer.decode([t])}»" if l != -100 else tokenizer.decode([t])
                  for t, l in list(zip(e["input_ids"], e["labels"]))[:limit]))

show_mask(encoded["train"][0])

<|doctor|>«What»« brings»« you»« in»« today»«?»«<|patient|>»I've got a cough<|doctor|>«When»« did»« the»« cough»« start»«?»«<|patient|>»Yesterday<|doctor|>«Any»« fever»«?»«<|patient|>»Yeah, a bit<|doctor|>«S»«ore»« throat»«?»«<|patient|>»Yeah, that too<|doctor|>«Short»«ness»« of»« breath»«?»«<|patient|>»Sometimes, yeah<|doctor|>«Running»« nose»«?»«<|patient|>»Yeah, it's runny<|doctor|>«C»«ough»«ing»« up»« ph»«leg»«m»«?»«<|patient|>»I've got coughing up phlegm too<|doctor|>«Thanks»« for»« your»« time»« today»«.»«<|diagnosis|>»«Ac»«ute»« bron»«ch»«itis»«;»« P»«neum»«onia»«;»« Ac»«ute»« sin»«us»«itis»«<|endoftext|>»


In [15]:
# Check: trained tokens decode to exactly the doctor turns, their end markers and the diagnosis line,
# and no patient text is ever trained.
by_id = {d["dialogue_id"]: d for d in dialogues}
markers = [ROLE_TOKENS["patient"], DIAGNOSIS_TOKEN, tokenizer.eos_token]

for s in SPLITS:
    for e in encoded[s]:
        d = by_id[e["dialogue_id"]]
        trained = tokenizer.decode([t for t, l in zip(e["input_ids"], e["labels"]) if l != -100],
                                   clean_up_tokenization_spaces=False)
        expected = "".join(t["text"] + (ROLE_TOKENS["patient"] if i < len(d["turns"]) - 1 else DIAGNOSIS_TOKEN)
                           for i, t in enumerate(d["turns"]) if t["role"] == "doctor")
        expected += "; ".join(d["diagnosis"]) + tokenizer.eos_token
        assert trained == expected, (e["dialogue_id"], trained[:200], expected[:200])
print("mask OK")

mask OK


## 7. Save the tokenized dataset

In [16]:
from datasets import Dataset, DatasetDict

tok_dir = OUT_DIR / "tokenized" / MODEL_SLUG
cols = ["dialogue_id", "input_ids", "attention_mask", "labels"]
dsd = DatasetDict({s: Dataset.from_list([{k: e[k] for k in cols} for e in encoded[s]]) for s in SPLITS})
dsd.save_to_disk(str(tok_dir / "dataset"))
tokenizer.save_pretrained(str(tok_dir / "tokenizer"))

prep_config = {
    "model_name": MODEL_NAME, "max_length": MAX_LEN, "train_role": "doctor",
    "role_tokens": ROLE_TOKENS, "extra_special_tokens": EXTRA_SPECIAL_TOKENS, "vocab_size": len(tokenizer),
    "stop_tokens": [ROLE_TOKENS["patient"], DIAGNOSIS_TOKEN, tokenizer.eos_token],
    "cards_source": str(CARDS_JSONL or "standin"), "dialogues_source": str(DIALOGUES_JSONL or "standin-nb-doctor"),
    "conditions": CONDITIONS, "cards_per_condition": CARDS_PER_CONDITION, "dialogues_per_card": DIALOGUES_PER_CARD,
    "max_questions": MAX_QUESTIONS, "seed": SEED,
    "counts": {s: len(encoded[s]) for s in SPLITS}, "dropped_too_long": {s: len(v) for s, v in too_long.items()},
    "rejected_by_filter": len(rejected),
}
(tok_dir / "prep_config.json").write_text(json.dumps(prep_config, indent=2), encoding="utf-8")
print("saved ->", tok_dir)
dsd

Saving the dataset (0/1 shards):   0%|          | 0/4827 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/201 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/201 [00:00<?, ? examples/s]

saved -> /home/kth/Documents/Github/project/model/data/processed/doctor/tokenized/gpt2


DatasetDict({
    train: Dataset({
        features: ['dialogue_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 4827
    })
    dev: Dataset({
        features: ['dialogue_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201
    })
    test: Dataset({
        features: ['dialogue_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201
    })
})

## 8. Next steps
- `04_doctor_train.ipynb` fine-tunes GPT-2 small and medium on this dataset (learning rate × model size grid) and keeps the best on dev.
- `05_doctor_eval.ipynb` evaluates every doctor in auto mode on the held-out test cards: naive Bayes, pretrained GPT-2, fine-tuned GPT-2, and LLaMA-3-8B / Mistral-7B with zero-shot, few-shot and CoT prompting.
- `CARDS_JSONL` / `DIALOGUES_JSONL` point at notebook 07's output. Other dialogue sources need `dialogue_id`, `card_id`, `split`, `disease`, `turns` (doctor first and last) and `diagnosis` (true disease first).